In [1]:
import os

from bs4 import BeautifulSoup
from dotenv import load_dotenv
from openai import OpenAI
import requests


In [2]:
load_dotenv()

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

anthropic_api_url = "https://api.anthropic.com/v1/"

openai = OpenAI(api_key=openai_api_key)
anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_api_url)

In [3]:
def fetch_website_contents(url: str) -> str | None:
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
    }
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return None
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return title + "\n\n" + text

In [4]:
def fetch_wikipedia_page(subject:str) -> str | None:
    url = f"https://en.wikipedia.org/wiki/{subject.lower().strip().replace(' ', '_').replace("-", "_")}"
    return fetch_website_contents(url)

In [18]:
city = "Paris"

In [ ]:
page = fetch_wikipedia_page(city)

In [ ]:
def summarize_text(text: str, model: str = "gpt-5.6-luna") -> str | None:
    system_prompt = """You are a helpful assistant that summarizes text in a concise, structured and compelling way,
    ignoring text that might be navigation related. 
    Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown."""
    user_prompt = f"Summarize the following text in a concise, structured and compelling way:\n\n{text}"
    response = openai.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
    )
    return response.choices[0].message.content

In [ ]:
if page:
    summarized_page = summarize_text(page)

In [ ]:
def find_out_the_main_spoken_language_in_city(city: str, model: str = "gpt-5.6-luna") -> str | None:
    system_prompt = """You are a helpful assistant that finds out the main language spoken in a city.
    Respond with the name of the language only, without any additional text."""
    user_prompt = f"What is the main language spoken in {city}?"
    response = openai.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
    )
    return response.choices[0].message.content

In [19]:
spoken_language = find_out_the_main_spoken_language_in_city(city)

In [ ]:
def translate_text(text: str, target_language: str, model: str = "gpt-5.6-luna") -> str | None:
    system_prompt = f"""You are a helpful assistant that translates text into {target_language}.
    Respond with the translated text only, without any additional text.
    Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown."""
    user_prompt = f"Translate the following text into {target_language}:\n\n{text}"
    response = openai.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}],
    )
    return response.choices[0].message.content

In [21]:
if summarized_page and spoken_language:
    translated_summary = translate_text(summarized_page, spoken_language)

In [22]:
translated_summary

'# Paris en un coup d’œil\n\n- Ce qu’est Paris\n  - Capitale et plus grande ville de France ; pôle mondial de la culture, de la finance, de la diplomatie, de la mode et de la gastronomie.\n  - Située sur la Seine, dans la région Île-de-France ; fait partie de la vaste aire métropolitaine parisienne (Grand Paris).\n\n- Quelques chiffres\n  - Population de la ville : environ 2,05 millions d’habitants (2026) ; l’empreinte métropolitaine et urbaine atteint environ 11 à 13 millions d’habitants dans l’aire élargie, selon le mode de calcul.\n  - Superficie : environ 105 km² pour la ville ; tissu urbain dense comptant environ 19 400 habitants par km².\n  - Structure administrative : 20 arrondissements ; la ville de Paris se trouve dans la région Île-de-France ; la Métropole du Grand Paris assure la coopération avec les banlieues proches et éloignées.\n  - Économie : PIB de la ville de Paris d’environ 280 milliards d’euros ; PIB de l’Île-de-France d’environ 865 milliards d’euros ; un cœur de l’